# SO-101 ACT Training on Colab T4 — All Three Tasks

Trains **pick → plug → pour** end-to-end in one session on a free T4 GPU.
Each policy is pushed back to your HuggingFace Hub immediately after its training
finishes, so if anything dies later you keep what's done.

**Expected wall-clock on T4 GPU at FULL_QUALITY=True:**
| Task | Steps | Wall-clock | Why |
|------|-------|------------|-----|
| pick | 25,000 | ~40-50 min | Easiest task, baseline |
| plug | 30,000 | ~60-75 min | Hardest task, biggest budget |
| pour | 20,000 | ~40-50 min | Medium-difficulty |
| **Total** | **75,000** | **~2.5-3 h** | T4 free tier is plenty |

If Colab's 12 h session window or compute-unit cap is a worry, set
`FULL_QUALITY=False` to drop to the compressed step counts (still on GPU; ~30-40 min total).

## Before you Run-All
1. **Runtime → Change runtime type → T4 GPU**.
2. The datasets `SurajCreation/so101_pick_v1`, `..._plug_v1`, `..._pour_v1` must already exist on the Hub. (They were pushed earlier.)
3. Have a HuggingFace **write** token from <https://huggingface.co/settings/tokens>. Paste it when the auth cell asks.

## 1. Configuration

In [ ]:
# === EDIT THESE ===
HF_USERNAME   = "SurajCreation"                      # owner of the dataset and policy repos
TASKS_TO_RUN  = ["pick", "plug", "pour"]              # any subset, in any order
FULL_QUALITY  = True                                  # True: full-size ACT @ many steps. False: compressed
PUSH_TO_HF    = True                                  # push trained policies back to your Hub (private repos)
POLICY_PRIVATE = True                                 # private model repos on HF

# === Probably leave alone ===
# Auto-resume from existing checkpoints if a Colab session was interrupted mid-training.
AUTO_RESUME   = True
# Colab's T4 has 16 GB VRAM; 8 fits the full-size ACT comfortably with 2 cameras at 480x640.
BATCH_SIZE    = 8
# Print a training-progress line every N steps. Lower = more verbose.
LOG_FREQ      = 200

# Per-task settings — match scripts/task{1,2,3}_env.sh on your Mac.
CONFIGS = {
    "pick": {
        "dataset_repo": f"{HF_USERNAME}/so101_pick_v1",
        "policy_repo":  f"{HF_USERNAME}/act_pick_v1",
        "run_name":     "act_pick_v1",
        "steps_quick":  8000,
        "steps_full":   25000,
        "save_freq":    2500,
        "chunk_size":   50,
        "kl_weight":    10.0,
    },
    "plug": {
        "dataset_repo": f"{HF_USERNAME}/so101_plug_v1",
        "policy_repo":  f"{HF_USERNAME}/act_plug_v1",
        "run_name":     "act_plug_v1",
        "steps_quick":  11000,
        "steps_full":   30000,
        "save_freq":    3000,
        "chunk_size":   80,
        "kl_weight":    20.0,
    },
    "pour": {
        "dataset_repo": f"{HF_USERNAME}/so101_pour_v1",
        "policy_repo":  f"{HF_USERNAME}/act_pour_v1",
        "run_name":     "act_pour_v1",
        "steps_quick":  5500,
        "steps_full":   20000,
        "save_freq":    2000,
        "chunk_size":   80,
        "kl_weight":    15.0,
    },
}

for t in TASKS_TO_RUN:
    assert t in CONFIGS, f"unknown task: {t}"

# Make HF avoid xet upload backend (same fix that worked on macOS).
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"User           : {HF_USERNAME}")
print(f"Tasks to train : {TASKS_TO_RUN}")
print(f"Quality        : {'FULL (full-size ACT)' if FULL_QUALITY else 'COMPRESSED (smaller ACT)'}")
print(f"Push to HF     : {PUSH_TO_HF}")
print(f"Auto-resume    : {AUTO_RESUME}")

## 2. Verify GPU — hard-fail if T4 not assigned

A common gotcha: forgetting to switch the runtime to GPU. Training would silently fall back to CPU and take 50× longer. We hard-stop here.

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        "nvidia-smi failed — no GPU is attached.\n"
        "Go to Runtime → Change runtime type → T4 GPU, save, then re-run this cell."
    )
print(result.stdout)

## 3. Install lerobot 0.5.1 and dependencies

Pinned to match the version on your Mac. This takes ~3-5 min the first time.

In [ ]:
%pip install -q --upgrade pip
%pip install -q "lerobot==0.5.1" "huggingface_hub>=0.27"

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch reports CUDA is not available even though nvidia-smi worked.\n"
        "Restart the runtime (Runtime → Restart runtime) and re-run from the top."
    )
print(f"PyTorch: {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"GPU     : {torch.cuda.get_device_name(0)} (capability {torch.cuda.get_device_capability(0)})")

## 4. Authenticate with HuggingFace

Use a **write**-scoped token from <https://huggingface.co/settings/tokens>.

In [ ]:
from huggingface_hub import login, whoami
login()
me = whoami()
print(f"Logged in as: {me['name']} (token type: {me.get('auth', {}).get('accessToken', {}).get('role', 'unknown')})")
if me["name"] != HF_USERNAME:
    print(f"WARNING: token belongs to '{me['name']}' but HF_USERNAME='{HF_USERNAME}'. Make sure you have access to those repos.")

## 5. Keepalive (fights Colab's 30-min idle disconnect)

Injects a tiny JS click on the "Connect" button every 5 min. Run this cell once and forget it.

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function ColabKeepAlive() {
  const btn = document.querySelector('colab-toolbar-button#connect');
  if (btn) { btn.click(); }
}
setInterval(ColabKeepAlive, 5 * 60 * 1000);
console.log('Colab keepalive armed (every 5 min)');
'''))
print("Keepalive armed. Tab can stay in background; please don't close it.")

## 6. Helper functions

In [ ]:
import os, sys, glob, shutil, subprocess, time
from pathlib import Path
from huggingface_hub import HfApi, create_repo, snapshot_download

OUT_BASE = Path("/content/outputs")
OUT_BASE.mkdir(parents=True, exist_ok=True)

def dataset_local_root(dataset_repo: str) -> str:
    """Where lerobot will read the dataset from on Colab disk."""
    return f"/root/.cache/huggingface/lerobot/{dataset_repo}"

def download_dataset(dataset_repo: str) -> str:
    """Pull the dataset snapshot from HF if not already present locally. Returns the local root."""
    local_root = dataset_local_root(dataset_repo)
    sentinel = Path(local_root) / "meta" / "info.json"
    if sentinel.exists():
        print(f"  [dataset] cached at {local_root}")
        return local_root
    print(f"  [dataset] downloading {dataset_repo} -> {local_root}")
    Path(local_root).mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=dataset_repo,
        repo_type="dataset",
        local_dir=local_root,
    )
    return local_root

def find_existing_checkpoint(out_dir: Path) -> Path | None:
    """Return the most recent checkpoint dir that contains train_config.json, or None."""
    last = out_dir / "checkpoints" / "last" / "pretrained_model" / "train_config.json"
    if last.exists():
        return last.parent
    candidates = sorted(out_dir.glob("checkpoints/*/pretrained_model/train_config.json"))
    return candidates[-1].parent if candidates else None

def stream_subprocess(cmd: list[str], log_path: Path) -> int:
    """Run cmd, stream stdout/stderr live to the cell output AND tee to log_path. Return exit code."""
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log = open(log_path, "a", buffering=1)
    log.write(f"\n\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
    log.write(" ".join(cmd) + "\n\n")
    log.flush()
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
    finally:
        proc.wait()
        log.close()
    return proc.returncode

def upload_policy(out_dir: Path, policy_repo: str) -> str | None:
    """Push the latest checkpoint's pretrained_model dir to the Hub."""
    if not PUSH_TO_HF:
        return None
    last_pretrained = out_dir / "checkpoints" / "last" / "pretrained_model"
    if not last_pretrained.exists():
        # Fallback: pick the highest-numbered checkpoint dir
        ckpts = sorted(out_dir.glob("checkpoints/*/pretrained_model"))
        if not ckpts:
            print(f"  [upload] no checkpoint to push for {policy_repo}")
            return None
        last_pretrained = ckpts[-1]
    print(f"  [upload] {last_pretrained} -> {policy_repo}")
    api = HfApi()
    create_repo(policy_repo, exist_ok=True, private=POLICY_PRIVATE)
    api.upload_folder(
        folder_path=str(last_pretrained),
        repo_id=policy_repo,
        repo_type="model",
        commit_message=f"ACT policy trained on Colab T4",
    )
    return f"https://huggingface.co/{policy_repo}"

def build_train_cmd(task: str, c: dict, dataset_root: str, out_dir: Path, resume_from: Path | None) -> list[str]:
    """Construct the lerobot-train CLI for one task. Honors AUTO_RESUME when a checkpoint exists."""
    if FULL_QUALITY:
        steps      = c["steps_full"]
        dim_model  = 512
        n_heads    = 8
        dim_ff     = 3200
    else:
        steps      = c["steps_quick"]
        dim_model  = 256
        n_heads    = 4
        dim_ff     = 1600

    if resume_from is not None:
        # Resume mode: load all config from train_config.json; only override steps if needed.
        return [
            "lerobot-train",
            f"--config_path={resume_from}/train_config.json",
            "--resume=true",
            f"--steps={steps}",
        ]

    return [
        "lerobot-train",
        "--policy.type=act",
        "--policy.device=cuda",
        "--policy.push_to_hub=false",
        f"--policy.chunk_size={c['chunk_size']}",
        f"--policy.n_action_steps={c['chunk_size']}",
        "--policy.use_vae=true",
        f"--policy.kl_weight={c['kl_weight']}",
        "--policy.optimizer_lr=1e-5",
        f"--policy.dim_model={dim_model}",
        f"--policy.n_heads={n_heads}",
        f"--policy.dim_feedforward={dim_ff}",
        f"--dataset.repo_id={c['dataset_repo']}",
        f"--dataset.root={dataset_root}",
        f"--batch_size={BATCH_SIZE}",
        f"--steps={steps}",
        f"--save_freq={c['save_freq']}",
        f"--log_freq={LOG_FREQ}",
        f"--output_dir={out_dir}",
        f"--job_name={c['run_name']}_colab",
        "--wandb.enable=false",
    ]

def train_one_task(task: str) -> dict:
    """Run the full pipeline for one task: download → train (or resume) → upload."""
    c = CONFIGS[task]
    print(f"\n{'='*70}")
    print(f"  TASK: {task.upper()}    run_name={c['run_name']}")
    print(f"  dataset_repo={c['dataset_repo']}")
    print(f"  policy_repo ={c['policy_repo']}")
    print(f"{'='*70}")

    out_dir = OUT_BASE / c["run_name"]

    dataset_root = download_dataset(c["dataset_repo"])

    resume_from = None
    if AUTO_RESUME and out_dir.exists():
        resume_from = find_existing_checkpoint(out_dir)
        if resume_from is not None:
            print(f"  [resume] continuing from {resume_from}")
        else:
            # output_dir exists but has no checkpoints — wipe it so lerobot doesn't refuse to start
            print(f"  [reset] {out_dir} exists but has no checkpoint — removing for fresh start")
            shutil.rmtree(out_dir)

    cmd = build_train_cmd(task, c, dataset_root, out_dir, resume_from)
    print(f"\n  $ {' '.join(cmd)}\n")

    log_path = out_dir / "training.log"
    t0 = time.time()
    rc = stream_subprocess(cmd, log_path)
    elapsed = time.time() - t0
    if rc != 0:
        return {"task": task, "ok": False, "elapsed_s": elapsed, "err": f"lerobot-train exited {rc}"}

    pushed_url = None
    try:
        pushed_url = upload_policy(out_dir, c["policy_repo"])
    except Exception as e:
        return {"task": task, "ok": True, "elapsed_s": elapsed, "upload_err": f"{type(e).__name__}: {e}"}

    return {"task": task, "ok": True, "elapsed_s": elapsed, "policy_url": pushed_url}

print("Helpers defined.")

## 7. Run all training

This is the long-running cell. Watch the log lines stream live. Each task pushes its trained policy to the Hub *immediately* after training, so partial progress is preserved if Colab disconnects later.

In [ ]:
import time
session_t0 = time.time()
RESULTS: list[dict] = []

for task in TASKS_TO_RUN:
    try:
        r = train_one_task(task)
    except Exception as e:
        r = {"task": task, "ok": False, "err": f"{type(e).__name__}: {e}"}
    RESULTS.append(r)
    print(f"\n  >>> finished {task}: {r}")

session_elapsed = time.time() - session_t0
print(f"\n\n{'='*70}")
print(f"SESSION SUMMARY  (total {session_elapsed/60:.1f} min)")
print(f"{'='*70}")
for r in RESULTS:
    if r.get("ok"):
        em = r.get("elapsed_s", 0)/60
        url = r.get("policy_url", "(not pushed)")
        print(f"  OK   {r['task']:5s}  {em:5.1f} min  {url}")
    else:
        print(f"  FAIL {r['task']:5s}  {r.get('err','?')}")

## 8. Pull the trained policies back to your Mac

After the cell above shows three OK lines, run **on your Mac** (not in Colab):

```bash
cd ~/Desktop/FInal_robothon
for task in pick plug pour; do
  .conda/bin/huggingface-cli download SurajCreation/act_${task}_v1 \
    --local-dir outputs/act_${task}_v1/checkpoints/last/pretrained_model
done
```

Then `./scripts/run_full_demo.sh` runs the autonomous Pick → Plug → Pour sequence with the cloud-trained policies.

## 9. (Optional) Inspect a checkpoint or download the zips manually

Skip this cell unless you want a local zip in case the HF push failed.

In [ ]:
import shutil
for task in TASKS_TO_RUN:
    c = CONFIGS[task]
    out_dir = OUT_BASE / c["run_name"]
    pretrained = out_dir / "checkpoints" / "last" / "pretrained_model"
    if not pretrained.exists():
        print(f"  [skip] no checkpoint for {task} at {pretrained}")
        continue
    zip_path = f"/content/{c['run_name']}.zip"
    shutil.make_archive(zip_path[:-4], "zip", pretrained)
    print(f"  [zip] {zip_path}  ({Path(zip_path).stat().st_size/1024/1024:.1f} MB)")
print("\nIn Colab's file browser (folder icon on the left), right-click each zip and pick Download.")